## Test Clock Propagation

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

### Monte-Carlo Simulation

In [ ]:
import os
import numpy as np
from concurrent.futures import ProcessPoolExecutor


class ClockNoise:
    def __init__(self, clock_model):
        self.clock_model = clock_model
        if clock_model == "OCXO":
            # For LCRNS
            self.sigma_wf = 6.2299445014e-13  # White frequency noise (s/sqrt(s))
            self.sigma_rw = 2.0129544799e-14  # Random walk frequency noise (s^(-1/2))
            self.sigma_rr = 7.0118586804e-28  # Random run frequency noise (s^(-3/2))
            self.sigma_wp = 1.4562174977e-19  # White phase noise (s)
            # self.sigma_z0 = 3.1943096158e-16  # Initial clock bias (s^(-1))
        else:
            raise ValueError(f"Unsupported clock model: {clock_model}")

    def get_process_noise(self, tau):
        s2_wf = self.sigma_wf**2
        s2_rw = self.sigma_rw**2
        s2_rr = self.sigma_rr**2

        q11 = s2_wf * tau + (s2_rw * tau**3) / 3 + (s2_rr * tau**5) / 20
        q12 = (s2_rw * tau**2) / 2 + (s2_rr * tau**4) / 8
        q13 = (s2_rr * tau**3) / 6
        q22 = s2_rw * tau + (s2_rr * tau**3) / 3
        q23 = (s2_rr * tau**2) / 2
        q33 = s2_rr * tau
        Q = np.array([[q11, q12, q13], [q12, q22, q23], [q13, q23, q33]])
        return Q

    def simulate_clock_bias(self, tspan, sigma_bias, sigma_drift):
        n = len(tspan)
        dt = np.diff(tspan, prepend=0)
        clk_bias = np.zeros((n, 3))  # [bias, drift, drift_rate]

        # sample initial clock bias
        clk_bias[0, 0] = np.random.normal(0, sigma_bias)
        clk_bias[0, 1] = np.random.normal(0, sigma_drift)

        for i in range(1, n):
            tau = dt[i]
            Q = self.get_process_noise(tau)
            w = np.random.multivariate_normal(mean=[0, 0, 0], cov=Q)
            Phi = np.array([[1, tau, 0.5 * tau**2], [0, 1, tau], [0, 0, 1]])
            clk_bias[i] = Phi @ clk_bias[i - 1] + w

        return clk_bias

In [ ]:
C = 299792458  # speed of light in m/s
n_mc = 100
lent_hrs = 15 * 24  # hours
sigma_clk_bias = 3.0 / C
sigma_clk_drift = 1e-3 / C

if lent_hrs < 24:
    tau = 1.0
    lent = lent_hrs * 3600  # sec
else:
    tau = 10.0
    lent = int(lent_hrs * 3600 // tau)  # number of time steps

clk_bias = np.zeros((n_mc, lent, 3))  # [bias, drift, drift_rate]
tspan = np.arange(0, lent * tau, tau)
clock_model = "OCXO"
clock_noise = ClockNoise(clock_model=clock_model)

In [ ]:
# Simulate clock bias for each Monte Carlo run
for mc_idx in tqdm(range(n_mc)):
    clk_bias[mc_idx, :, :] = clock_noise.simulate_clock_bias(
        tspan, sigma_clk_bias, sigma_clk_drift
    )

In [ ]:
# Plotting code here
import matplotlib.pyplot as plt
import os

convert_to_meters = True  # speed of light in m/s
if convert_to_meters:
    clk_bias[:, :, 0] *= 299792458  # convert bias from seconds to meters
    clk_bias[:, :, 1] *= 299792458  # convert drift from s/s to m/s
    clk_bias[:, :, 2] *= 299792458  # convert drift rate from s/s² to m/s²
    labels = ["Clock Bias (m)", "Clock Drift (m/s)", "Clock Drift Rate (m/s²)"]
else:
    labels = ["Clock Bias (s)", "Clock Drift (s/s)", "Clock Drift Rate (s/s²)"]

fig, axes = plt.subplots(3, 1, figsize=(7, 7), sharex=True)
i = 0
for label in labels:
    axes[i].plot(tspan / 3600, clk_bias[:, :, i].T, color="gray")
    axes[i].set_ylabel(label)
    axes[i].set_xlabel("Time (hours)")
    axes[i].grid(True)
    i += 1

# print std at final epoch
final_bias_var = np.std(clk_bias[:, -1, 0])
final_drift_var = np.std(clk_bias[:, -1, 1])
final_drift_rate_var = np.std(clk_bias[:, -1, 2])
print(f"Final clock bias std: {final_bias_var:.3e} {'m' if convert_to_meters else 's'}")
print(
    f"Final clock drift std: {final_drift_var:.3e} {'m/s' if convert_to_meters else 's/s'}"
)
print(
    f"Final clock drift rate std: {final_drift_rate_var:.3e} {'m/s²' if convert_to_meters else 's/s²'}"
)

# save to file
plt.tight_layout()
basedir = "/Users/keidaiiiyama/Documents/sw_navlab/LuPNT-private/output/ephemeris/"
figdir = basedir + "clock_noise/"
if not os.path.exists(figdir):
    os.makedirs(figdir)
plt.savefig(
    figdir + "clock_noise_{}_lent{}hrs.pdf".format(clock_model, lent_hrs), dpi=300
)
plt.show()